In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%cd /kaggle/working
!git clone https://github.com/ThaoTo2806/CodeFuse123.git
%cd /kaggle/working/CodeFuse123

In [ ]:
!pip install -q transformers==4.46.1 pandas

In [4]:
from pathlib import Path
import hashlib
import json
import os

import pandas as pd
from transformers import AutoTokenizer

# ============================================================
# 1. CAU HINH DUONG DAN
# ============================================================

DATA_DIR = Path("/kaggle/working/CodeFuse123/dataset")

TRAIN_PATH = DATA_DIR / "train.json"
TEST_PATH = DATA_DIR / "test.json"

# Neu da upload tokenizer/model vao Kaggle Input,
# sua MODEL_DIR thanh duong dan thu muc chua tokenizer.
MODEL_DIR = None

MODEL_ID = "meta-llama/Llama-2-7b-hf"
MODEL_REVISION_LOCK = None

MAX_LENGTH = 512

OUTPUT_DIR = Path("/kaggle/working/improve_step1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. KIEM TRA VA DOC DATASET
# ============================================================

if not TRAIN_PATH.is_file():
    raise FileNotFoundError(
        f"Khong tim thay train.json: {TRAIN_PATH}"
    )

if not TEST_PATH.is_file():
    raise FileNotFoundError(
        f"Khong tim thay test.json: {TEST_PATH}"
    )

with TRAIN_PATH.open("r", encoding="utf-8") as f:
    train_rows = json.load(f)

with TEST_PATH.open("r", encoding="utf-8") as f:
    test_rows = json.load(f)

if not isinstance(train_rows, list) or not isinstance(test_rows, list):
    raise ValueError(
        "train.json va test.json phai chua JSON array (list)."
    )

if not train_rows or not test_rows:
    raise ValueError("Train hoac test dang rong.")

required_keys = {"change", "summaries", "type"}

for split_name, rows in [
    ("train", train_rows),
    ("test", test_rows),
]:
    for index, row in enumerate(rows):
        if not isinstance(row, dict):
            raise ValueError(
                f"{split_name}[{index}] khong phai JSON object."
            )

        missing = required_keys - set(row.keys())
        if missing:
            raise ValueError(
                f"{split_name}[{index}] thieu cac truong: {missing}. "
                f"Cac truong hien co: {list(row.keys())}"
            )

        if not isinstance(row["change"], str):
            raise ValueError(
                f"{split_name}[{index}]['change'] phai la chuoi."
            )

        if not isinstance(row["summaries"], dict):
            raise ValueError(
                f"{split_name}[{index}]['summaries'] phai la object."
            )

        if "en" not in row["summaries"]:
            raise ValueError(
                f"{split_name}[{index}]['summaries'] thieu truong 'en'."
            )

print("=" * 60)
print("DATASET")
print("=" * 60)
print(f"Dataset directory: {DATA_DIR}")
print(f"Train file: {TRAIN_PATH}")
print(f"Test file:  {TEST_PATH}")
print(f"So mau train: {len(train_rows):,}")
print(f"So mau test:  {len(test_rows):,}")


# ============================================================
# 3. TAO CHSUM PROMPT
# ============================================================

def make_chsum_prompt(diff):
    return (
        "Please generate a commit message for the following diff:\n\n"
        f"```diff\n{diff}\n```\n"
    )


# ============================================================
# 4. LOAD TOKENIZER
# ============================================================

hf_token = None

if MODEL_DIR is not None:
    model_source = str(Path(MODEL_DIR))

    if not Path(model_source).is_dir():
        raise FileNotFoundError(
            f"MODEL_DIR khong ton tai: {model_source}"
        )

    tokenizer_kwargs = {"local_files_only": True}

else:
    model_source = MODEL_ID

    # Lay token tu Kaggle Secrets hoac environment variable
    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = UserSecretsClient().get_secret("kaggle-llama")

    except Exception:
        hf_token = os.environ.get("kaggle-llama")

    if not hf_token:
        raise RuntimeError(
            "Khong tim thay HF_TOKEN. Hay them HF_TOKEN vao "
            "Kaggle Secrets, hoac dat MODEL_DIR den thu muc "
            "tokenizer da upload vao Kaggle."
        )

    tokenizer_kwargs = {"token": hf_token}

    if MODEL_REVISION_LOCK:
        tokenizer_kwargs["revision"] = MODEL_REVISION_LOCK

tokenizer = AutoTokenizer.from_pretrained(
    model_source,
    **tokenizer_kwargs,
)

resolved_revision = tokenizer.init_kwargs.get("_commit_hash")


# ============================================================
# 5. TINH FINGERPRINT TOKENIZER
# ============================================================

if MODEL_DIR is not None:
    tokenizer_file_names = {
        "tokenizer_config.json",
        "special_tokens_map.json",
        "tokenizer.json",
        "tokenizer.model",
        "vocab.json",
        "merges.txt",
        "added_tokens.json",
    }

    fingerprint = hashlib.sha256()
    local_path = Path(MODEL_DIR)

    for file_path in sorted(
        path
        for path in local_path.rglob("*")
        if path.is_file()
        and path.name in tokenizer_file_names
    ):
        relative_name = file_path.relative_to(
            local_path
        ).as_posix()

        fingerprint.update(relative_name.encode("utf-8"))
        fingerprint.update(file_path.read_bytes())

    tokenizer_fingerprint = fingerprint.hexdigest()
    resolved_revision = resolved_revision or "local-upload"

else:
    tokenizer_fingerprint = None

print("\n" + "=" * 60)
print("TOKENIZER")
print("=" * 60)
print(f"Tokenizer source: {model_source}")
print(f"Resolved revision: {resolved_revision}")
print(f"Max length: {MAX_LENGTH}")


# ============================================================
# 6. THONG KE SO TOKEN CHO TRAIN VA TEST
# ============================================================

summary_rows = []
sample_rows = []

for split_name, rows in [
    ("train", train_rows),
    ("test", test_rows),
]:
    print(f"\nDang xu ly {split_name}: {len(rows):,} mau...")

    for index, row in enumerate(rows):
        diff = row["change"]

        # Token cua diff goc
        diff_token_count = len(
            tokenizer(
                diff,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )

        # Token cua prompt CHSUM
        chsum_prompt = make_chsum_prompt(diff)

        prompt_token_count = len(
            tokenizer(
                chsum_prompt,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )

        sample_rows.append({
            "split": split_name,
            "index": index,
            "diff_tokens": diff_token_count,
            "chsum_prompt_tokens": prompt_token_count,
        })

    # Lay cac mau cua split hien tai
    split_samples = [
        item
        for item in sample_rows
        if item["split"] == split_name
    ]

    for input_name, length_key in [
        ("diff_only", "diff_tokens"),
        ("chsum_prompt", "chsum_prompt_tokens"),
    ]:
        lengths = pd.Series(
            [item[length_key] for item in split_samples],
            dtype="int64",
        )

        over_limit = int((lengths > MAX_LENGTH).sum())

        summary_rows.append({
            "split": split_name,
            "input_view": input_name,
            "samples": len(lengths),
            "max_length": MAX_LENGTH,
            "over_limit_count": over_limit,
            "over_limit_rate": (
                over_limit / len(lengths)
                if len(lengths) > 0 else 0.0
            ),
            "mean_tokens": float(lengths.mean()),
            "median_tokens": float(lengths.median()),
            "p90_tokens": float(lengths.quantile(0.90)),
            "max_tokens": int(lengths.max()),
            "model_id": model_source,
            "model_revision": resolved_revision,
        })

summary = pd.DataFrame(summary_rows)
sample_df = pd.DataFrame(sample_rows)


# ============================================================
# 7. LUU KET QUA CSV
# ============================================================

summary_path = OUTPUT_DIR / "prompt_length_summary.csv"
sample_path = OUTPUT_DIR / "prompt_token_lengths.csv"

summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8",
)

sample_df.to_csv(
    sample_path,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 8. LUU METADATA
# ============================================================

metadata = {
    "model_id": model_source,
    "model_revision": resolved_revision,
    "model_revision_lock_used": MODEL_REVISION_LOCK,
    "tokenizer_fingerprint_sha256": tokenizer_fingerprint,
    "dataset_dir": str(DATA_DIR),
    "train_path": str(TRAIN_PATH),
    "test_path": str(TEST_PATH),
    "train_samples": len(train_rows),
    "test_samples": len(test_rows),
    "max_length": MAX_LENGTH,
    "views": {
        "diff_only": (
            "Raw code diff with tokenizer special tokens."
        ),
        "chsum_prompt": (
            "CHSUM wrapper around the raw code diff."
        ),
    },
}

metadata_path = OUTPUT_DIR / "tokenizer_audit_metadata.json"

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 9. HIEN THI KET QUA
# ============================================================

print("\n" + "=" * 60)
print("TOKEN LENGTH SUMMARY")
print("=" * 60)

display(summary)

print("\n" + "=" * 60)
print("HOAN TAT")
print("=" * 60)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Summary CSV: {summary_path}")
print(f"Sample CSV:  {sample_path}")
print(f"Metadata:    {metadata_path}")


DATASET
Dataset directory: /kaggle/working/CodeFuse123/dataset
Train file: /kaggle/working/CodeFuse123/dataset/train.json
Test file:  /kaggle/working/CodeFuse123/dataset/test.json
So mau train: 4,611
So mau test:  518


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]


TOKENIZER
Tokenizer source: meta-llama/Llama-2-7b-hf
Resolved revision: None
Max length: 512

Dang xu ly train: 4,611 mau...

Dang xu ly test: 518 mau...

TOKEN LENGTH SUMMARY


,split,input_view,samples,max_length,over_limit_count,over_limit_rate,mean_tokens,median_tokens,p90_tokens,max_tokens,model_id,model_revision
0,train,diff_only,4611,512,322,0.069833,341.492952,316.0,481.0,2808,meta-llama/Llama-2-7b-hf,None
1,train,chsum_prompt,4611,512,392,0.085014,359.492952,334.0,499.0,2826,meta-llama/Llama-2-7b-hf,None
2,test,diff_only,518,512,34,0.065637,334.799228,310.0,475.3,1102,meta-llama/Llama-2-7b-hf,None
3,test,chsum_prompt,518,512,40,0.077220,352.799228,328.0,493.3,1120,meta-llama/Llama-2-7b-hf,None



HOAN TAT
Output directory: /kaggle/working/improve_step1
Summary CSV: /kaggle/working/improve_step1/prompt_length_summary.csv
Sample CSV:  /kaggle/working/improve_step1/prompt_token_lengths.csv
Metadata:    /kaggle/working/improve_step1/tokenizer_audit_metadata.json


In [5]:
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/improve_step1_results",
    "zip",
    root_dir="/kaggle/working",
    base_dir="improve_step1",
)
print("ZIP kết quả:", zip_path)

ZIP kết quả: /kaggle/working/improve_step1_results.zip
